In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import zipfile

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

assert (PROJECT_ROOT / "PROJECT_PLAN.md").exists(), (
    "Make sure this notebook is saved in the project's notebooks directory"
)

print("Project root:", PROJECT_ROOT)

Project root: /Users/celestine/Desktop/intern/Customer Churn Project/financial-complaint-intelligence


In [2]:
def prepare_cfpb_archive(
    zip_path,
    source_id,
    source_label,
    download_url,
    project_root,
):
    """Extract a single CSV archive and record its source metadata."""
    zip_path = Path(zip_path)
    project_root = Path(project_root)

    if not zip_path.is_file():
        raise FileNotFoundError(f"Archive not found: {zip_path}")

    # Compute the archive fingerprint without loading it into memory
    sha256 = hashlib.sha256()
    with zip_path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(chunk)

    with zipfile.ZipFile(zip_path) as archive:
        csv_members = [
            item for item in archive.infolist()
            if not item.is_dir()
            and item.filename.lower().endswith(".csv")
        ]

        if len(csv_members) != 1:
            raise ValueError("Expected exactly one CSV file in the archive")

        member = csv_members[0]
        csv_path = zip_path.parent / Path(member.filename).name
        temp_path = csv_path.with_suffix(".csv.part")

        # Publish the extracted file only after extraction succeeds
        with archive.open(member) as source:
            with temp_path.open("wb") as destination:
                shutil.copyfileobj(
                    source, destination, length=1024 * 1024
                )

        if temp_path.stat().st_size != member.file_size:
            raise ValueError("Extracted file size does not match archive metadata")

        temp_path.replace(csv_path)

    manifest = {
        "dataset": "CFPB Consumer Complaint Narratives Archive",
        "source_id": source_id,
        "source_label": source_label,
        "download_url": download_url,
        "local_file": zip_path.relative_to(project_root).as_posix(),
        "size_bytes": zip_path.stat().st_size,
        "sha256": sha256.hexdigest(),
        "archive_members": [{
            "name": member.filename,
            "uncompressed_size_bytes": member.file_size,
        }],
        "manifest_created_at_utc": datetime.now(timezone.utc).isoformat(),
        "date_coverage_verified": False,
    }

    manifest_path = (
        project_root / "data" / "manifests" / f"{source_id}.json"
    )
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )

    return csv_path, manifest_path

In [3]:
CSV_PATH_2024, MANIFEST_PATH_2024 = prepare_cfpb_archive(
    zip_path=(
        PROJECT_ROOT
        / "data"
        / "raw"
        / "CCDB_Export_8_November_2024_through_December_2024.zip"
    ),
    source_id="cfpb_2024_nov_dec",
    source_label="November 2024 through December 2024",
    download_url=(
        "https://files.consumerfinance.gov/f/documents/"
        "CCDB_Export_8_November_2024_through_December_2024.zip"
    ),
    project_root=PROJECT_ROOT,
)

print("CSV:", CSV_PATH_2024.relative_to(PROJECT_ROOT))
print("CSV size (bytes):", CSV_PATH_2024.stat().st_size)
print("Manifest:", MANIFEST_PATH_2024.relative_to(PROJECT_ROOT))

CSV: data/raw/CCDB_Export_8_November_2024_through_December_2024.csv
CSV size (bytes): 347911485
Manifest: data/manifests/cfpb_2024_nov_dec.json


In [4]:
def audit_source_csv(csv_path, project_root):
    """Audit a source CSV using the project's saved SQL queries."""
    csv_path = Path(csv_path)
    sql_dir = Path(project_root) / "sql"

    audit_query = (
        sql_dir / "01_initial_audit.sql"
    ).read_text(encoding="utf-8")

    product_query = (
        sql_dir / "02_product_summary.sql"
    ).read_text(encoding="utf-8")

    # Use a separate connection for each source file
    with duckdb.connect() as connection:
        source = connection.read_csv(
            str(csv_path),
            header=True,
            all_varchar=True,
        )

        columns = source.columns
        source.create_view("complaints_raw")

        audit = connection.execute(audit_query).df()
        products = connection.execute(product_query).df()

    return audit, products, columns

In [5]:
audit_2024, products_2024, columns_2024 = audit_source_csv(
    CSV_PATH_2024,
    PROJECT_ROOT,
)

CSV_PATH_2025 = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "CCDB_Export_9_January_2025_through_February_2025.csv"
)

# Read only the header of the previously audited file
columns_2025 = pd.read_csv(
    CSV_PATH_2025,
    nrows=0,
).columns.tolist()

print("Same column names and order:",
      columns_2024 == columns_2025)

print("Columns missing from the new file:",
      sorted(set(columns_2025) - set(columns_2024)))

print("Additional columns in the new file:",
      sorted(set(columns_2024) - set(columns_2025)))

print("\nSource audit:")
display(audit_2024.T.rename(columns={0: "value"}))

print("\nCredit card subset:")
display(
    products_2024.loc[
        products_2024["product"] == "Credit card"
    ]
)

Same column names and order: True
Columns missing from the new file: []
Additional columns in the new file: []

Source audit:


,value
total_rows,591058
distinct_complaint_ids,591058
missing_complaint_ids,0
earliest_received_date,2024-11-01 00:00:00
latest_received_date,2024-12-31 00:00:00
missing_received_dates,0
unparsed_received_dates,0
rows_with_narrative,154916
narrative_coverage_pct,26.21



Credit card subset:


,product,total_rows,rows_with_narrative,narrative_coverage_pct
2,Credit card,12792,5527,43.21


In [6]:
audit_2024.to_csv(
    PROJECT_ROOT / "reports" / "cfpb_2024_nov_dec_audit.csv",
    index=False,
)

products_2024.to_csv(
    PROJECT_ROOT / "reports" / "cfpb_2024_nov_dec_products.csv",
    index=False,
)

print("Source audit and product summary saved.")

Source audit and product summary saved.


In [7]:
row = audit_2024.iloc[0]

assert int(row["missing_received_dates"]) == 0
assert int(row["unparsed_received_dates"]) == 0

observed_start = row["earliest_received_date"].strftime("%Y-%m-%d")
observed_end = row["latest_received_date"].strftime("%Y-%m-%d")

assert observed_start == "2024-11-01"
assert observed_end == "2024-12-31"

manifest_2024 = json.loads(
    MANIFEST_PATH_2024.read_text(encoding="utf-8")
)

manifest_2024["date_coverage_verified"] = True
manifest_2024["observed_date_coverage"] = {
    "earliest_received_date": observed_start,
    "latest_received_date": observed_end,
    "missing_received_dates": 0,
    "unparsed_received_dates": 0,
}
manifest_2024["observed_counts"] = {
    "total_rows": int(row["total_rows"]),
    "distinct_complaint_ids": int(row["distinct_complaint_ids"]),
    "rows_with_narrative": int(row["rows_with_narrative"]),
}

MANIFEST_PATH_2024.write_text(
    json.dumps(manifest_2024, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("Source manifest updated with verified receipt-date coverage.")

Source manifest updated with verified receipt-date coverage.


In [8]:
training_con = duckdb.connect()

training_con.read_csv(
    str(CSV_PATH_2024),
    header=True,
    all_varchar=True,
).create_view("source_2024_nov_dec", replace=True)

training_con.read_csv(
    str(CSV_PATH_2025),
    header=True,
    all_varchar=True,
).create_view("source_2025_jan_feb", replace=True)

print("Two source views created.")

Two source views created.


In [9]:
overlap_sql = """
WITH ids_2024 AS (
    SELECT DISTINCT
        NULLIF(TRIM("Complaint ID"), '') AS complaint_id
    FROM source_2024_nov_dec
),
ids_2025 AS (
    SELECT DISTINCT
        NULLIF(TRIM("Complaint ID"), '') AS complaint_id
    FROM source_2025_jan_feb
)
SELECT
    COUNT(*) AS overlapping_complaint_ids
FROM ids_2024
INNER JOIN ids_2025 USING (complaint_id)
"""

overlap_summary = training_con.execute(overlap_sql).df()

display(overlap_summary)

overlap_summary.to_csv(
    PROJECT_ROOT / "reports" / "source_id_overlap.csv",
    index=False,
)

(PROJECT_ROOT / "sql" / "10_source_id_overlap.sql").write_text(
    overlap_sql.strip() + "\n",
    encoding="utf-8",
)

,overlapping_complaint_ids
0,0


354

In [10]:
additional_sources = [
    {
        "source_id": "cfpb_2024_apr_jul",
        "source_label": "April 2024 through July 2024",
        "download_url": (
            "https://files.consumerfinance.gov/f/documents/"
            "CCDB_Export_6_April_2024_through_July_2024.zip"
        ),
    },
    {
        "source_id": "cfpb_2024_aug_oct",
        "source_label": "August 2024 through October 2024",
        "download_url": (
            "https://files.consumerfinance.gov/f/documents/"
            "CCDB_Export_7_August_2024_through_October_2024.zip"
        ),
    },
]

In [11]:
prepared_sources = {}

for source in additional_sources:
    filename = source["download_url"].rsplit("/", 1)[-1]
    zip_path = PROJECT_ROOT / "data" / "raw" / filename

    csv_path, manifest_path = prepare_cfpb_archive(
        zip_path=zip_path,
        source_id=source["source_id"],
        source_label=source["source_label"],
        download_url=source["download_url"],
        project_root=PROJECT_ROOT,
    )

    prepared_sources[source["source_id"]] = {
        "csv_path": csv_path,
        "manifest_path": manifest_path,
    }

    print("Source:", source["source_id"])
    print("CSV:", csv_path.name)
    print("CSV size (bytes):", csv_path.stat().st_size)
    print("Manifest:", manifest_path.relative_to(PROJECT_ROOT))
    print("-" * 60)

Source: cfpb_2024_apr_jul
CSV: CCDB_Export_6_April_2024_through_July_2024.csv
CSV size (bytes): 517112772
Manifest: data/manifests/cfpb_2024_apr_jul.json
------------------------------------------------------------
Source: cfpb_2024_aug_oct
CSV: CCDB_Export_7_August_2024_through_October_2024.csv
CSV size (bytes): 484574093
Manifest: data/manifests/cfpb_2024_aug_oct.json
------------------------------------------------------------


In [12]:
training_sources = [
    {
        "source_id": "cfpb_2024_apr_jul",
        "csv_path": prepared_sources["cfpb_2024_apr_jul"]["csv_path"],
        "expected_start": "2024-04-01",
        "expected_end": "2024-07-31",
    },
    {
        "source_id": "cfpb_2024_aug_oct",
        "csv_path": prepared_sources["cfpb_2024_aug_oct"]["csv_path"],
        "expected_start": "2024-08-01",
        "expected_end": "2024-10-31",
    },
    {
        "source_id": "cfpb_2024_nov_dec",
        "csv_path": CSV_PATH_2024,
        "expected_start": "2024-11-01",
        "expected_end": "2024-12-31",
    },
    {
        "source_id": "cfpb_2025_jan_feb",
        "csv_path": CSV_PATH_2025,
        "expected_start": "2025-01-01",
        "expected_end": "2025-02-28",
    },
]

In [13]:
audit_results = []
product_results = []

for source in training_sources:
    audit, products, columns = audit_source_csv(
        source["csv_path"],
        PROJECT_ROOT,
    )

    if columns != columns_2025:
        raise ValueError(
            f"Column names or order differ: {source['source_id']}"
        )

    row = audit.iloc[0]
    actual_start = row["earliest_received_date"].strftime("%Y-%m-%d")
    actual_end = row["latest_received_date"].strftime("%Y-%m-%d")

    date_bounds_match = (
        actual_start == source["expected_start"]
        and actual_end == source["expected_end"]
    )

    audit = audit.assign(
        source_id=source["source_id"],
        date_bounds_match=date_bounds_match,
        internal_duplicate_id_rows=(
            int(row["total_rows"])
            - int(row["missing_complaint_ids"])
            - int(row["distinct_complaint_ids"])
        ),
    )

    products = products.assign(source_id=source["source_id"])

    audit_results.append(audit)
    product_results.append(products)

    print("Audited:", source["source_id"])

training_source_audits = pd.concat(
    audit_results,
    ignore_index=True,
)

training_source_products = pd.concat(
    product_results,
    ignore_index=True,
)

Audited: cfpb_2024_apr_jul
Audited: cfpb_2024_aug_oct
Audited: cfpb_2024_nov_dec
Audited: cfpb_2025_jan_feb


In [14]:
audit_columns = [
    "source_id",
    "total_rows",
    "distinct_complaint_ids",
    "missing_complaint_ids",
    "internal_duplicate_id_rows",
    "earliest_received_date",
    "latest_received_date",
    "missing_received_dates",
    "unparsed_received_dates",
    "rows_with_narrative",
    "date_bounds_match",
]

print("Training source audits:")
print(
    training_source_audits[audit_columns].to_string(index=False)
)

credit_card_sources = training_source_products.loc[
    training_source_products["product"] == "Credit card",
    [
        "source_id",
        "total_rows",
        "rows_with_narrative",
        "narrative_coverage_pct",
    ],
].copy()

print("\nCredit card counts by source:")
print(credit_card_sources.to_string(index=False))

training_source_audits.to_csv(
    PROJECT_ROOT / "reports" / "training_source_audits.csv",
    index=False,
)

training_source_products.to_csv(
    PROJECT_ROOT / "reports" / "training_source_products.csv",
    index=False,
)

print("\nBatch audit reports saved.")

Training source audits:
        source_id  total_rows  distinct_complaint_ids  missing_complaint_ids  internal_duplicate_id_rows earliest_received_date latest_received_date  missing_received_dates  unparsed_received_dates  rows_with_narrative  date_bounds_match
cfpb_2024_apr_jul      839903                  839903                      0                           0             2024-04-01           2024-07-31                       0                        0               276332               True
cfpb_2024_aug_oct      824514                  824514                      0                           0             2024-08-01           2024-10-31                       0                        0               229944               True
cfpb_2024_nov_dec      591058                  591058                      0                           0             2024-11-01           2024-12-31                       0                        0               154916               True
cfpb_2025_jan_feb      8

In [15]:
training_csv_paths = [
    str(source["csv_path"])
    for source in training_sources
]

training_con.read_csv(
    training_csv_paths,
    header=True,
    all_varchar=True,
    filename=True,
).create_view("training_raw", replace=True)

print("Combined training view created.")
print("Source files:", len(training_csv_paths))

Combined training view created.
Source files: 4


In [16]:
training_integrity_sql = """
WITH normalized AS (
    SELECT
        NULLIF(TRIM("Complaint ID"), '') AS complaint_id
    FROM training_raw
)
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT complaint_id) AS distinct_complaint_ids,
    COUNT(*) FILTER (
        WHERE complaint_id IS NULL
    ) AS missing_complaint_ids,
    COUNT(complaint_id)
        - COUNT(DISTINCT complaint_id) AS duplicate_id_rows
FROM normalized
"""

training_integrity = training_con.execute(
    training_integrity_sql
).df()

display(training_integrity.T.rename(columns={0: "value"}))

expected_rows = int(training_source_audits["total_rows"].sum())
result = training_integrity.iloc[0]

assert int(result["total_rows"]) == expected_rows, (
    "Combined row count differs from the sum of source row counts"
)
assert int(result["missing_complaint_ids"]) == 0, (
    "Missing complaint IDs found"
)
assert int(result["duplicate_id_rows"]) == 0, (
    "Duplicate complaint IDs found across source files"
)

print("Combined row count and ID checks passed.")

,value
total_rows,3066494
distinct_complaint_ids,3066494
missing_complaint_ids,0
duplicate_id_rows,0


Combined row count and ID checks passed.


In [17]:
monthly_sql = """
WITH credit_card AS (
    SELECT
        CAST(
            TRY_STRPTIME(
                TRIM("Date received"),
                ['%Y-%m-%d', '%m/%d/%Y']
            ) AS DATE
        ) AS received_date,
        NULLIF(
            TRIM("Consumer complaint narrative"), ''
        ) AS narrative
    FROM training_raw
    WHERE TRIM("Product") = 'Credit card'
)
SELECT
    STRFTIME(received_date, '%Y-%m') AS month,
    COUNT(*) AS total_rows,
    COUNT(narrative) AS rows_with_narrative,
    ROUND(
        100.0 * COUNT(narrative) / COUNT(*),
        2
    ) AS narrative_coverage_pct
FROM credit_card
GROUP BY month
ORDER BY month
"""

training_monthly = training_con.execute(monthly_sql).df()

print(training_monthly.to_string(index=False))

expected_months = pd.period_range(
    "2024-04", "2025-02", freq="M"
).astype(str).tolist()

actual_months = training_monthly["month"].tolist()

assert actual_months == expected_months, (
    "Monthly coverage differs from the planned training period"
)

assert int(training_monthly["total_rows"].sum()) == int(
    credit_card_sources["total_rows"].sum()
)

assert int(training_monthly["rows_with_narrative"].sum()) == int(
    credit_card_sources["rows_with_narrative"].sum()
)

print("\nMonthly coverage and credit card totals verified.")

  month  total_rows  rows_with_narrative  narrative_coverage_pct
2024-04        6676                 3793                   56.82
2024-05        6760                 3580                   52.96
2024-06        5535                 2484                   44.88
2024-07        6192                 2542                   41.05
2024-08        7355                 2784                   37.85
2024-09        7185                 2681                   37.31
2024-10        7176                 2768                   38.57
2024-11        6095                 2630                   43.15
2024-12        6697                 2897                   43.26
2025-01        9073                 3992                   44.00
2025-02        6213                 2966                   47.74

Monthly coverage and credit card totals verified.


In [18]:
outputs = [
    (
        "training_integrity",
        training_integrity_sql,
        training_integrity,
    ),
    (
        "training_monthly",
        monthly_sql,
        training_monthly,
    ),
]

for name, query, table in outputs:
    (PROJECT_ROOT / "sql" / f"{name}.sql").write_text(
        query.strip() + "\n",
        encoding="utf-8",
    )

    table.to_csv(
        PROJECT_ROOT / "reports" / f"{name}.csv",
        index=False,
    )

print("Combined audit queries and reports saved.")

Combined audit queries and reports saved.


In [19]:
extraction_sql = """
SELECT
    NULLIF(TRIM("Complaint ID"), '') AS complaint_id,
    CAST(
        TRY_STRPTIME(
            TRIM("Date received"),
            ['%Y-%m-%d', '%m/%d/%Y']
        ) AS DATE
    ) AS received_date,
    TRIM("Product") AS product,
    NULLIF(TRIM("Sub-product"), '') AS sub_product,
    NULLIF(TRIM("Issue"), '') AS issue,
    NULLIF(TRIM("Sub-issue"), '') AS sub_issue,
    NULLIF(
        TRIM("Consumer complaint narrative"), ''
    ) AS narrative,
    NULLIF(TRIM("Company"), '') AS company,
    NULLIF(TRIM("Submitted via"), '') AS submitted_via,
    REGEXP_EXTRACT(filename, '[^/]+$', 0) AS source_file
FROM training_raw
WHERE TRIM("Product") = 'Credit card'
"""

credit_card_training_all = training_con.execute(
    extraction_sql
).df()

credit_card_training_text = credit_card_training_all.loc[
    credit_card_training_all["narrative"].notna()
].copy()

print("All credit card records:", len(credit_card_training_all))
print("Credit card records with text:", len(credit_card_training_text))

All credit card records: 74957
Credit card records with text: 33117


In [20]:
expected_all_rows = int(credit_card_sources["total_rows"].sum())
expected_text_rows = int(
    credit_card_sources["rows_with_narrative"].sum()
)

assert len(credit_card_training_all) == expected_all_rows
assert len(credit_card_training_text) == expected_text_rows

assert credit_card_training_all["complaint_id"].notna().all()
assert credit_card_training_all["complaint_id"].is_unique
assert credit_card_training_all["received_date"].notna().all()

assert credit_card_training_all["received_date"].min() >= pd.Timestamp(
    "2024-04-01"
)
assert credit_card_training_all["received_date"].max() <= pd.Timestamp(
    "2025-02-28"
)

extraction_summary = pd.DataFrame([
    {
        "dataset": "credit_card_training_all",
        "rows": len(credit_card_training_all),
        "missing_issue_rows": int(
            credit_card_training_all["issue"].isna().sum()
        ),
    },
    {
        "dataset": "credit_card_training_text",
        "rows": len(credit_card_training_text),
        "missing_issue_rows": int(
            credit_card_training_text["issue"].isna().sum()
        ),
    },
])

display(extraction_summary)

print("Extraction count, ID, and date checks passed.")

,dataset,rows,missing_issue_rows
0,credit_card_training_all,74957,0
1,credit_card_training_text,33117,0


Extraction count, ID, and date checks passed.


In [21]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "credit_card_training_all": credit_card_training_all,
    "credit_card_training_text": credit_card_training_text,
}

for name, table in datasets.items():
    path = PROCESSED_DIR / f"{name}.parquet"
    table.to_parquet(path, index=False, compression="zstd")

    # Read only the ID column to verify the saved row count
    saved_ids = pd.read_parquet(
        path,
        columns=["complaint_id"],
    )

    assert len(saved_ids) == len(table), (
        f"Saved row count differs for {name}"
    )

    print("Saved:", path.relative_to(PROJECT_ROOT))
    print("Rows verified:", len(saved_ids))
    print("Size (MB):", round(path.stat().st_size / 1_000_000, 2))
    print("-" * 60)

extraction_summary.to_csv(
    PROJECT_ROOT / "reports" / "training_extraction_summary.csv",
    index=False,
)

(PROJECT_ROOT / "sql" / "training_credit_card_extraction.sql").write_text(
    extraction_sql.strip() + "\n",
    encoding="utf-8",
)

Saved: data/processed/credit_card_training_all.parquet
Rows verified: 74957
Size (MB): 13.62
------------------------------------------------------------
Saved: data/processed/credit_card_training_text.parquet
Rows verified: 33117
Size (MB): 13.29
------------------------------------------------------------


673